# Textbook Tutor on Google Colab

Runs your full app in Google's free cloud — no credit card, login with your Gmail.

**How to use:**
1. Runtime menu → **Run all** (or press Ctrl/Cmd+F9)
2. When asked, choose your Gmail account and **Allow** Drive access
3. Wait for the last cell — it prints the shareable link
4. Share that link with students. Open it with `/admin` to manage (default password `admin123`)

**Every session you start fresh, the app restores your database and OCR models from Google Drive**, so nothing is lost between runs.

**Limits (free Colab):** a session runs at most ~12 hours, and ends if the tab is closed. To restart, just reopen this notebook and Run all again — data is safe on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

REPO = "/content/textbook-tutor"
LOCAL_DATA = "/content/tt-data"
DRIVE_DATA = "/content/drive/MyDrive/textbook-tutor-data"
os.makedirs(REPO, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)

if not os.listdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/nobnoob001-ops/textbook-tutor.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "pull"], capture_output=True)

os.environ["DATA_DIR"] = LOCAL_DATA
os.environ["EASYOCR_MODULE_PATH"] = "/content/.EasyOCR"
print("Repo ready. Data will persist to", DRIVE_DATA)

In [ ]:
import subprocess
subprocess.run("apt-get update -qq && apt-get install -qq -y tesseract-ocr tesseract-ocr-ben poppler-utils > /dev/null 2>&1", shell=True)
subprocess.run("pip -q install easyocr fastapi uvicorn httpx pypdf pdf2image pytesseract 2>&1 | tail -n 2", shell=True)
print("Dependencies installed.")

In [ ]:
import glob, os, shutil

def _cp(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

src_db = os.path.join(DRIVE_DATA, "tutor.db")
if os.path.exists(src_db):
    shutil.copy(src_db, os.path.join(LOCAL_DATA, "tutor.db"))
    print("Restored database from Drive.")

for p in glob.glob(os.path.join(DRIVE_DATA, ".EasyOCR", "**", "*"), recursive=True):
    if os.path.isfile(p):
        rel = os.path.relpath(p, os.path.join(DRIVE_DATA, ".EasyOCR"))
        _cp(p, os.path.join("/content/.EasyOCR", rel))
print("Data restored.")

In [ ]:
import os, sqlite3, threading, time

def backup_db():
    src = os.path.join(LOCAL_DATA, "tutor.db")
    if not os.path.exists(src):
        return
    tmp = os.path.join(DRIVE_DATA, "tutor.db.tmp")
    dst = os.path.join(DRIVE_DATA, "tutor.db")
    try:
        a = sqlite3.connect(src)
        b = sqlite3.connect(tmp)
        a.backup(b)
        b.close()
        a.close()
        os.replace(tmp, dst)
    except Exception as e:
        print("backup error:", e)

def _loop(interval):
    while True:
        time.sleep(interval)
        backup_db()

threading.Thread(target=_loop, args=(60,), daemon=True).start()
backup_db()
print("Autosave running: every 60s -> Google Drive")

In [ ]:
import os, subprocess, time, httpx

os.chdir(REPO)
subprocess.run(["pkill", "-f", "run.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

env = dict(os.environ, HOST="0.0.0.0", PORT="8080")
with open("/content/tt-server.log", "w") as log:
    proc = subprocess.Popen(
        ["python", "run.py"],
        stdout=log, stderr=subprocess.STDOUT, env=env,
    )

up = False
for _ in range(60):
    time.sleep(2)
    try:
        r = httpx.get("http://localhost:8080/", timeout=3)
        if r.status_code == 200:
            up = True
            break
    except Exception:
        pass
print("Server up:", up)
if not up:
    print(open("/content/tt-server.log").read()[-2000:])

In [ ]:
import glob, os, shutil, subprocess

models_dir = "/content/.EasyOCR"
if not os.path.isdir(os.path.join(models_dir, "model")):
    print("Downloading OCR models (one-time, ~2 min)...")
    subprocess.run(["python", "-c", "import easyocr; easyocr.Reader(['bn','en'], gpu=False)"], check=True)
    for p in glob.glob(os.path.join(models_dir, "**", "*"), recursive=True):
        if os.path.isfile(p):
            rel = os.path.relpath(p, models_dir)
            dst = os.path.join(DRIVE_DATA, ".EasyOCR", rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(p, dst)
    print("OCR models ready + cached to Drive.")
else:
    print("OCR models already present.")

In [ ]:
import os, re, subprocess, time

def ensure_cloudflared():
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "/usr/local/bin/cloudflared"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

ensure_cloudflared()
with open("/content/tt-tunnel.log", "w") as log:
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT)

URL = None
for _ in range(60):
    time.sleep(2)
    try:
        text = open("/content/tt-tunnel.log").read()
    except Exception:
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        URL = m.group(0)
        break
print("STUDENT LINK:", URL)
print("ADMIN LINK:", URL + "/admin")
if not URL:
    print(open("/content/tt-tunnel.log").read()[-2000:])

In [ ]:
import time, httpx

print("")
print("==================== SHARE THIS LINK WITH STUDENTS ====================")
print("  " + URL)
print("  Admin (password): " + URL + "/admin")
print("=========================================================================")
print("This cell keeps the session alive. Close the tab to stop the server.")
print("Your data is saved to Google Drive every 60 seconds.")
while True:
    time.sleep(60)
    backup_db()
    try:
        httpx.get("http://localhost:8080/", timeout=5)
    except Exception:
        pass

**Done!** Keep the last cell running. If the link ever stops working, rerun the whole notebook (Runtime → Run all).